# Gold: Data Quality Checks
Runs after the gold tables are rebuilt and fails the run if any check finds bad rows.

Each check is a query that returns the rows breaking a rule, so a passing check returns nothing. The checks cover the issues fixed in silver (duplicate customers, product versions multiplying sales, inconsistent sales amounts) so they can't come back unnoticed.

#Define Checks

In [0]:
# name -> query returning the violating rows. Tables are referenced as schema.table (current catalog: workspace).
CHECKS = {
    # Dimensions: one row per business key, surrogate keys unique.
    "dim_customers: customer_id is unique and not null": """
        SELECT customer_id FROM gold.dim_customers
        GROUP BY customer_id HAVING count(*) > 1 OR customer_id IS NULL
    """,
    "dim_customers: customer_key is unique": """
        SELECT customer_key FROM gold.dim_customers GROUP BY customer_key HAVING count(*) > 1
    """,
    "dim_products: one current row per product_number": """
        SELECT product_number FROM gold.dim_products
        GROUP BY product_number HAVING count(*) > 1 OR product_number IS NULL
    """,
    "dim_products: product_key is unique": """
        SELECT product_key FROM gold.dim_products GROUP BY product_key HAVING count(*) > 1
    """,
    # Fact: the joins must neither drop nor multiply sales lines.
    "fact_sales: row count matches silver.crm_sales": """
        SELECT f.n AS fact_rows, s.n AS silver_rows
        FROM (SELECT count(*) AS n FROM gold.fact_sales) f
        CROSS JOIN (SELECT count(*) AS n FROM silver.crm_sales) s
        WHERE f.n <> s.n
    """,
    "fact_sales: every sale has a product and a customer": """
        SELECT order_number FROM gold.fact_sales WHERE product_key IS NULL OR customer_key IS NULL
    """,
    "fact_sales: sales_amount = quantity * price, all positive": """
        SELECT order_number FROM gold.fact_sales
        WHERE price IS NULL OR price <= 0 OR sales_amount <> quantity * price
    """,
}

#Run Checks

In [0]:
failures = []
for name, query in CHECKS.items():
    bad = spark.sql(query)
    n_bad = bad.count()
    print(f"{'PASS' if n_bad == 0 else 'FAIL'}  {name}" + (f"  ({n_bad} rows)" if n_bad else ""))
    if n_bad:
        bad.limit(10).display()
        failures.append(name)

# Raising makes the notebook (and gold_orchestration / a Job running it) fail instead of publishing bad data.
if failures:
    raise AssertionError(f"{len(failures)} data quality check(s) failed: {failures}")